In [ ]:
Setup & Install

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'requests', 'pandas', 'numpy', 'matplotlib', 'seaborn'])

import pandas as pd
import numpy as np
import requests
import os

# Create folder structure
folders = [
    'data/raw',
    'data/processed',
    'data/db',
    'notebooks',
    'scripts',
    'sql',
    'dashboard',
    'reports'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ All libraries imported!")
print("✅ Folder structure created!")
print("\nFolders created:")
for f in folders:
    print(f"  ✅ {f}")

✅ All libraries imported!
✅ Folder structure created!

Folders created:
  ✅ data/raw
  ✅ data/processed
  ✅ data/db
  ✅ notebooks
  ✅ scripts
  ✅ sql
  ✅ dashboard
  ✅ reports


In [ ]:
Fetch live NAV from mfapi.in

In [2]:
import requests
import pandas as pd

# Fetch live NAV for 5 key mutual fund schemes
schemes = {
    'SBI Bluechip': 119551,
    'ICICI Bluechip': 120503,
    'Nippon Large Cap': 118632,
    'Axis Bluechip': 119092,
    'Kotak Bluechip': 120841
}

nav_data = []

for scheme_name, code in schemes.items():
    url = f"https://api.mfapi.in/mf/{code}"
    response = requests.get(url)
    data = response.json()
    
    latest_nav = data['data'][0]
    nav_data.append({
        'Scheme': scheme_name,
        'Code': code,
        'Date': latest_nav['date'],
        'NAV': float(latest_nav['nav'])
    })
    print(f"✅ {scheme_name}: ₹{latest_nav['nav']}")

df_nav = pd.DataFrame(nav_data)
df_nav.to_csv('data/raw/live_nav.csv', index=False)

print("\n=== LIVE NAV DATA ===")
print(df_nav)
print("\n✅ Live NAV saved to data/raw/live_nav.csv")

✅ SBI Bluechip: ₹104.70250
✅ ICICI Bluechip: ₹103.09480
✅ Nippon Large Cap: ₹97.19440
✅ Axis Bluechip: ₹6156.75320
✅ Kotak Bluechip: ₹246.98140

=== LIVE NAV DATA ===
             Scheme    Code        Date        NAV
0      SBI Bluechip  119551  01-06-2026   104.7025
1    ICICI Bluechip  120503  01-06-2026   103.0948
2  Nippon Large Cap  118632  01-06-2026    97.1944
3     Axis Bluechip  119092  01-06-2026  6156.7532
4    Kotak Bluechip  120841  01-06-2026   246.9814

✅ Live NAV saved to data/raw/live_nav.csv


In [ ]:
Load Fund Master Data

In [3]:
# Fetch Fund Master Data
url = "https://api.mfapi.in/mf/search?q=bluechip"
response = requests.get(url)
funds = response.json()

# Create DataFrame
fund_master = pd.DataFrame(funds)
print("=== FUND MASTER DATA ===")
print(fund_master.head(10))
print("\nShape:", fund_master.shape)

# Unique fund houses
print("\n=== UNIQUE FUND HOUSES ===")
print(fund_master['schemeName'].str.split('-').str[0].unique()[:10])

# Save to CSV
fund_master.to_csv('data/raw/fund_master.csv', index=False)
print("\n✅ Fund Master saved!")

# Validate AMFI codes
print("\n=== VALIDATING AMFI CODES ===")
for scheme, code in schemes.items():
    exists = code in fund_master['schemeCode'].values
    print(f"{scheme} ({code}): {'✅ Valid' if exists else '❌ Invalid'}")

print("\n✅ Data Quality Check Complete!")

=== FUND MASTER DATA ===
   schemeCode                                         schemeName
0      120186  ICICI Prudential US Bluechip Equity Fund - Dir...
1      120185  ICICI Prudential US Bluechip Equity Fund - Dir...
2      117620  ICICI Prudential US Bluechip Equity Fund - Growth
3      117619    ICICI Prudential US Bluechip Equity Fund - IDCW
4      103455  UTI Bluechip Flexicap Fund - Regular Plan - Di...
5      103457  UTI Bluechip Flexicap Fund - Regular Plan - Gr...
6      108467  ICICI Prudential Bluechip Fund - Institutional...
7      111381   Principal Emerging Bluechip Fund - Growth Option
8      111382  Principal Emerging Bluechip Fund - Half Yearly...
9      119440  Principal Emerging Bluechip Fund - Direct Plan...

Shape: (15, 2)

=== UNIQUE FUND HOUSES ===
['ICICI Prudential US Bluechip Equity Fund ' 'UTI Bluechip Flexicap Fund '
 'ICICI Prudential Bluechip Fund ' 'Principal Emerging Bluechip Fund '
 'ICICI Prudential Large Cap Fund (erstwhile Bluechip Fund)  ']

✅ Fun

In [ ]:
Fetch NAV History

In [4]:
# Fetch NAV History for all 5 schemes
import time

all_nav_history = []

for scheme_name, code in schemes.items():
    url = f"https://api.mfapi.in/mf/{code}"
    response = requests.get(url)
    data = response.json()
    
    # Get last 30 days NAV
    nav_history = data['data'][:30]
    
    for nav in nav_history:
        all_nav_history.append({
            'Scheme': scheme_name,
            'Code': code,
            'Date': nav['date'],
            'NAV': float(nav['nav'])
        })
    
    print(f"✅ {scheme_name}: {len(nav_history)} records fetched")
    time.sleep(0.5)

# Save to CSV
df_history = pd.DataFrame(all_nav_history)
df_history.to_csv('data/raw/nav_history.csv', index=False)

print(f"\n=== NAV HISTORY ===")
print(df_history.head(10))
print(f"\nTotal Records: {len(df_history)}")
print("✅ NAV History saved to data/raw/nav_history.csv")
print("\n✅ Day 1 Data Ingestion Complete!")

✅ SBI Bluechip: 30 records fetched
✅ ICICI Bluechip: 30 records fetched
✅ Nippon Large Cap: 30 records fetched
✅ Axis Bluechip: 30 records fetched
✅ Kotak Bluechip: 30 records fetched

=== NAV HISTORY ===
         Scheme    Code        Date       NAV
0  SBI Bluechip  119551  01-06-2026  104.7025
1  SBI Bluechip  119551  29-05-2026  104.6570
2  SBI Bluechip  119551  27-05-2026  104.5833
3  SBI Bluechip  119551  26-05-2026  104.5269
4  SBI Bluechip  119551  25-05-2026  104.5215
5  SBI Bluechip  119551  22-05-2026  104.3467
6  SBI Bluechip  119551  21-05-2026  104.2850
7  SBI Bluechip  119551  20-05-2026  104.3826
8  SBI Bluechip  119551  19-05-2026  104.4216
9  SBI Bluechip  119551  18-05-2026  104.3791

Total Records: 150
✅ NAV History saved to data/raw/nav_history.csv

✅ Day 1 Data Ingestion Complete!


In [ ]:
Create requirments.txt

In [5]:
requirements = """pandas
numpy
matplotlib
seaborn
requests
scipy
jupyter
sqlalchemy
plotly
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("✅ requirements.txt created!")

# Create README.md
readme = """# Bluestock Mutual Fund Analytics - Capstone Project

## Project Overview
This project analyzes mutual fund data from AMFI/mfapi.in to provide insights on NAV trends, fund performance, and investment recommendations.

## Project Structure
- data/raw/ - Original downloaded files
- data/processed/ - Cleaned, merged CSVs
- data/db/ - SQLite database
- notebooks/ - Jupyter notebooks
- scripts/ - Python scripts
- sql/ - SQL schema and queries
- dashboard/ - Power BI dashboard
- reports/ - Final report and presentation

## Day 1 Deliverables
- 01_data_ingestion.ipynb
- data/raw/live_nav.csv
- data/raw/nav_history.csv
- data/raw/fund_master.csv
- requirements.txt

## Team
- Srushti Deshmukh
- Bluestock Fintech Internship 2026
"""

with open('README.md', 'w') as f:
    f.write(readme)

print("✅ README.md created!")
print("\n✅ Day 1 Capstone Project COMPLETE!")
print("Deliverables ready:")
print("  ✅ 01_data_ingestion.ipynb")
print("  ✅ data/raw/live_nav.csv")
print("  ✅ data/raw/nav_history.csv")
print("  ✅ data/raw/fund_master.csv")
print("  ✅ requirements.txt")
print("  ✅ README.md")

✅ requirements.txt created!
✅ README.md created!

✅ Day 1 Capstone Project COMPLETE!
Deliverables ready:
  ✅ 01_data_ingestion.ipynb
  ✅ data/raw/live_nav.csv
  ✅ data/raw/nav_history.csv
  ✅ data/raw/fund_master.csv
  ✅ requirements.txt
  ✅ README.md
